# Controller T4 frozen development comparison

Private dataset + T4 GPU + Internet. Frozen Qwen/NF4, CPU FP32 reranker, prefill256. No QA gold, selection or final inputs.
One run of development 200 per dataset, seven paired groups. C1 cached. No tuning or training. Gold scored offline.

In [ ]:
import os, sys, json, hashlib, shutil, subprocess, tarfile, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable CUDA GPU and Internet'
WORK=Path('/kaggle/working')
BUNDLE=Path('/tmp')/('controller_'+'c82c9d2fdf4f')/'controller_t3_gpu_v1'
OUTPUT=WORK/'controller_t4_development_v1_outputs'
assert not OUTPUT.exists(), 'Preserve previous outputs'
def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for part in iter(lambda:f.read(8*1024*1024),b''):h.update(part)
    return h.hexdigest()
if not BUNDLE.exists():
    archives=list(Path('/kaggle/input').rglob('controller_t3_gpu_v1.tar.gz'))
    indexes=[p for p in Path('/kaggle/input').rglob('bundle_checksums.json') if file_sha(p)=='c82c9d2fdf4fcc3448088d6627f4d080305617a510f921f6451e0e417b82c1cb']
    BUNDLE.parent.mkdir(parents=True,exist_ok=True)
    if indexes:
        assert len(indexes)==1
        shutil.copytree(indexes[0].parent,BUNDLE)
    else:
        assert len(archives)==1 and file_sha(archives[0])=='ddd51973e9921402f914cf0ccb942e5450a8ca402f2bbcc58d1af2825415cffe', 'Attach the private code dataset'
        with tarfile.open(archives[0]) as t:t.extractall(BUNDLE.parent,filter='data')
index=BUNDLE/'bundle_checksums.json'
assert file_sha(index)=='c82c9d2fdf4fcc3448088d6627f4d080305617a510f921f6451e0e417b82c1cb'
for rel,digest in json.loads(index.read_text()).items():
    assert not Path(rel).is_absolute() and '..' not in Path(rel).parts
    assert file_sha(BUNDLE/rel)==digest,rel
print('Frozen code and development inputs verified; no gold/selection/final input files')


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r',str(BUNDLE/'requirements-controller-t3.txt')],check=True)
subprocess.run([sys.executable,'-m','unittest','tests.test_controller_t3_execution','tests.test_controller_t3_evaluation',
                'tests.test_controller_integration','tests.test_controller_evaluation','tests.test_controller_t3_gate','-q'],cwd=BUNDLE,check=True)
subprocess.run([sys.executable,'scripts/run_controller_t4.py','--preflight','--output-dir',str(OUTPUT)],cwd=BUNDLE,check=True)


In [ ]:
# Bind immutable assets outside /kaggle/working so output exports do not duplicate GB resources.
markers=[p for p in Path('/kaggle/input').rglob('resource_bundle_v1.json') if json.loads(p.read_text()).get('schema_version')=='controller_t4_resource_bundle_v1']
if markers:
    assert len(markers)==1
    RESOURCES=markers[0].parent
else:
    archives=list(Path('/kaggle/input').rglob('controller_t4_resource_bundle_v1.tar.gz'))
    assert len(archives)==1 and file_sha(archives[0])=='d8e84e8b0c4c472ab0c0fe23e3e79d24645456ab3322cb93a24fefe77272b5f4'
    TEMP=Path('/tmp/controller_resource_unpack'); TEMP.mkdir(exist_ok=False)
    with tarfile.open(archives[0]) as t:t.extractall(TEMP,filter='data')
    RESOURCES=TEMP/'controller_t4_resources_v1'
manifest=json.loads((RESOURCES/'resource_bundle_v1.json').read_text())
expected={'data/processed/indexes/nq_train_bge_base_offsets.npy': {'bytes': 9793080, 'sha256': '3e819f87f04f6069e27f0ba12f89493834cbb597e2e0d2d599aee585b63d2e5b'}, 'data/processed/indexes/nq_train_bge_base_ids.jsonl': {'bytes': 37947689, 'sha256': '7d5d51f81ae3f89f4d7407b46b71f5c42804e418042e49c647435457f3715581'}, 'data/processed/indexes/nq_train_bge_base.index': {'bytes': 3760493613, 'sha256': 'cd94b5cd2d5fec4736ba16197e2431a87d347640cd1c79da6a6daa2cac9bfd5d'}, 'data/processed/retrieval/nq_train_corpus.jsonl': {'bytes': 782900757, 'sha256': 'c4ec2cc61aa091bd4cd6492b7cbdc53c8265395d9ac1b9b6307ac748daf6940d'}, 'data/processed/indexes/nq_train_bge_metadata.json': {'bytes': 4134, 'sha256': 'c6ed43b627833dd6592a964e5e03eebfd1fad3ba627f2c66a568ad94e7914fbd'}, 'artifacts/controller/nq_mapping_full_v1/validation.json': {'bytes': 3220, 'sha256': 'cdff67e6dcdab1d7751623402a99b04a1cf80285b50c30f5daff93e840668ff1'}, 'data/processed/controller_resources/fever_v1/fever_corpus.jsonl': {'bytes': 3522640169, 'sha256': '29dde612e13203ec37a566c2284d5e312eabc8e66277b972a6e643511fe65fd1'}, 'data/processed/controller_resources/fever_v1/fever_bge_ivfpq.index': {'bytes': 401921924, 'sha256': 'a000e7e994a4da47ec1d7af9371eab3c9e4872b2eeefb2f786cb102b4528046b'}, 'data/processed/controller_resources/fever_v1/fever_bge_ivfpq_ids.jsonl': {'bytes': 213760540, 'sha256': '9bcecd97153871da40edd6ebdee8b149a976174cf929685f77f5a34deba2539a'}, 'data/processed/controller_resources/fever_v1/fever_bge_ivfpq_meta.json': {'bytes': 601, 'sha256': 'f4146debafd05ff4b78d8788265c9df8ceb59462d40883beb6ef70eda6e6c2be'}, 'data/processed/controller_resources/fever_v1/fever_bge_ivfpq_offsets.npy': {'bytes': 43168976, 'sha256': 'd764edc54e65a2f7d7255a2efbe7ec04c64b29758c15d965ee336a83202b485a'}, 'artifacts/controller/fever_resources_v1/import_validation.json': {'bytes': 2156, 'sha256': '3f7eb4d05dea50a07241d2a5a63f9392ddda6ca0209b239c191159dc896b3c64'}, 'artifacts/controller/fever_resources_v1/structure_validation.json': {'bytes': 3270, 'sha256': '49ffb987228e84a13ff85254983cea327114a4e898c2da530e4864e4b41a7d72'}, 'artifacts/controller/fever_provider_v1/query_replay.json': {'bytes': 8920, 'sha256': 'a576c7a3e12bf0a450f913431a87bf8d72726932cd08eab9b1e3c147955e634f'}}
assert manifest['files']==expected and manifest['contains_gold'] is False
for rel,info in expected.items():
    source=RESOURCES/rel
    assert source.stat().st_size==info['bytes'] and file_sha(source)==info['sha256'],rel
    target=BUNDLE/rel;target.parent.mkdir(parents=True,exist_ok=True)
    assert not target.exists()
    target.symlink_to(source)
receipts=[p for p in Path('/kaggle/input').rglob('engineering_receipt.json') if file_sha(p)=='5894f9f86dd87ef1537cacf0c9f703ca0005747a87f6d0077b9694bc06ffbf60']
assert len(receipts)==1, 'Attach the accepted CUDA receipt'
shutil.copyfile(receipts[0],BUNDLE/'engineering_receipt.json')
print('Original retrieval assets and accepted CUDA receipt verified')


In [ ]:
command=[sys.executable,'-u','scripts/run_controller_t4.py','--output-dir',str(OUTPUT)]
command+=['--engineering-receipt', 'engineering_receipt.json']
log=WORK/'controller_t3_t4_run.log'
with log.open('x') as handle:
    p=subprocess.Popen(command,cwd=BUNDLE,env={**os.environ,'PYTHONUNBUFFERED':'1'},stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout:
        print(line,end='',flush=True);handle.write(line);handle.flush()
    returncode=p.wait()
OUTPUT.mkdir(exist_ok=True)
shutil.copyfile(log,OUTPUT/'run.log')
shutil.copyfile(BUNDLE/'execution_manifest.json',OUTPUT/'input_execution_manifest.json')
print('Process return code:',returncode,'; exporting all partial or completed outputs')


In [ ]:
assert OUTPUT.exists()
checks={str(p.relative_to(OUTPUT)):file_sha(p) for p in OUTPUT.rglob('*') if p.is_file() and p.name!='output_checksums.json'}
(OUTPUT/'output_checksums.json').write_text(json.dumps(checks,indent=2))
archive=WORK/(OUTPUT.name+'.zip')
with zipfile.ZipFile(archive,'x',compression=zipfile.ZIP_DEFLATED,strict_timestamps=False) as z:
    for p in sorted(OUTPUT.rglob('*')):
        if p.is_file():z.write(p,str(p.relative_to(OUTPUT)))
print('Exported:',archive,'SHA256:',file_sha(archive))
assert returncode==0, 'Preserve failed run and partial outputs; no semantic retry'
